# Gemma 4 E2B — ARDB unified page understanding (bbox + text, one forward pass) (Unsloth, LoRA/QLoRA, T4)

Fine-tunes `unsloth/gemma-4-E2B-it` (4-bit base + LoRA adapters = QLoRA) on
`Soxavin/ardb-sft-v5` — **one task, one forward pass**: full page image in, one JSON
list out, where every region carries its `box_2d`, `label`, AND its transcribed `text`
together:

```json
[
  {"box_2d": [15,15,90,90], "label": "Picture", "text": ""},
  {"box_2d": [21,91,51,266], "label": "Page-Furniture", "text": "ធនាគារ ARDB"},
  {"box_2d": [111,15,885,984], "label": "Table", "text": "| ២៣ | ... |"},
  {"box_2d": [889,17,983,385], "label": "Section-Header", "text": "ខែមករា ថ្ងៃទី26"}
]
```

This supersedes an earlier v1 design that split layout detection (bbox only) and
transcription (text only, on pre-cropped ground-truth regions) into two separate tasks/
forward passes — that never taught the model to connect "this region" to "this text" from
one image itself. v2 fixes that: every row is a full page, and the model must locate AND
read each region in a single call.

**v5** (`Soxavin/ardb-sft-v5`, 47 non-frozen documents / 128 pages: 101 train / 9 validation /
18 test) stratifies the train/validation/test split by structural template era on top of
date-clustering, so both bulletin layouts are represented in every split — see that dataset's
README for the full breakdown. Training images additionally get light train-only augmentation
(brightness/contrast jitter + occasional blur, see the cell below `convert_to_conversation`) —
validation/test images are never touched by it.

**Steps:** Runtime ▸ Change runtime type ▸ **T4 GPU** → run all cells with `SMOKE_TEST = True`
first (10 examples, ~a few minutes) to confirm nothing crashes/OOMs → then set
`SMOKE_TEST = False` and run again for the full training run. To sweep epoch count, change
`EPOCHS` in the cell below and rerun from there — each value pushes its adapter to its own
repo so runs don't overwrite each other.

In [ ]:
%%capture
import os, re
# Page targets vary a lot in length (a page with a long table + title + letterhead vs. a
# sparser one), which produces enough distinct compiled shapes to hit Dynamo's
# recompile_limit (FailOnRecompileLimitHit) -- and raising the limit didn't help, since
# Unsloth's own compilation patch resets it internally after model load, silently undoing
# any override made beforehand. Disabling the auto-compiler is Unsloth's documented fix for
# this case (https://unsloth.ai/docs/basics/unsloth-environment-flags) -- must be set before
# unsloth is imported anywhere. (The official Gemma4 (E2B) Vision notebook instead just sets
# torch._dynamo.config.recompile_limit = 64 after install -- that's the one deliberate
# deviation from the official recipe in this cell; everything else below is matched line for
# line, including the previously-missing `torchcodec` install added back below.)
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
# torchcodec: present in the official notebook's install cell, was missing here. Added back
# unpinned/without --no-deps, matching the official recipe exactly (Qwen's notebook pins
# torchcodec==0.7.0 instead, but that's a different, separately-verified recipe -- not
# assumed transferable here).
!pip install torchcodec
!pip install --no-deps --upgrade timm  # Gemma 4 vision


In [ ]:
from unsloth import FastVisionModel
import torch

# device_map={"": 0}: forces the whole model onto GPU 0 rather than letting accelerate's
# automatic device_map inference decide per-module placement. A 5.2B-param model in 4-bit
# (~2.6GB of weights) easily fits a 14.5GB T4, so any auto-computed dispatch onto "cpu"/
# "disk" is a false positive (from an accelerate/bitsandbytes version mismatch, since cell 1
# installs those unpinned) -- bitsandbytes' 4-bit quantizer hard-errors on that combination
# (ValueError: "Some modules are dispatched on the CPU or the disk").
model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-E2B-it",
    load_in_4bit = True,               # QLoRA: 4-bit base weights
    use_gradient_checkpointing = "unsloth",
    device_map = {"": 0},
)

In [ ]:
# LoRA adapters on top of the frozen 4-bit base. r/alpha kept modest (32) since T4 has
# only 16GB — raise if the smoke test comfortably fits and you want more capacity.
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 32,
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
    target_modules = "all-linear",
)

## Data

One flat dataset, one schema — `image` + `instruction` + `text` (the JSON list of regions).
No config merging or column alignment needed anymore.

In [ ]:
SMOKE_TEST = True  # flip to False only after a smoke run has completed without errors

# Epoch sweep knob (see SFTConfig cell below): Runs 1-3 on v2 (66 rows) found JSON-parse-
# failure rate got WORSE as epochs went up (2 epochs: 1/9 failures; 5 epochs: 3/9, then
# 9/9 on a repeat) even as training loss kept dropping cleanly -- a small-dataset
# overfitting signature. v5 has a different row count/split shape than v2, so this value
# should be re-swept empirically (try 2, 3, 5, ...) rather than assumed -- change this and
# rerun from here; each value pushes its adapter to its own repo (see the push-to-hub cell)
# so sweep runs don't overwrite each other. Log each run's result in
# eval/gemma_finetune_runs.csv before changing this again.
EPOCHS = 3

from datasets import load_dataset

_REPO_ID = "Soxavin/ardb-sft-v5"

dataset = load_dataset(_REPO_ID, split="train")
val_dataset = load_dataset(_REPO_ID, split="validation")

if SMOKE_TEST:
    dataset = dataset.select(range(min(10, len(dataset))))
    val_dataset = val_dataset.select(range(min(5, len(val_dataset))))

print(f"train rows: {len(dataset)}, validation rows: {len(val_dataset)}")

In [ ]:
import random
from PIL import Image, ImageEnhance, ImageFilter

# Train-only augmentation: brightness/contrast jitter + light blur, applied only to
# converted_dataset below (never val_dataset -- the eval cells read val_dataset's images
# directly, never through convert_to_conversation, so there's no shared code path that
# could leak augmentation into eval). Deliberately no rotation/affine/scale/crop: box_2d
# targets are ground-truth coordinates the model must reproduce, and any geometric
# transform here would need every box_2d in the target JSON re-projected to match, which
# this pass doesn't do -- table-row/column alignment is exactly what the model needs to
# learn, and a naive rotate/crop would degrade it first. Ranges are roughly half of
# experiments/khmer_crnn/train.py's word-crop augmentation, since a whole page table has
# more surface area for a shifted brightness curve to blow out thin grid lines than an
# isolated word-crop does.
_aug_rng = random.Random(3407)

def augment_image(img: Image.Image, rng: random.Random) -> Image.Image:
    img = ImageEnhance.Brightness(img).enhance(1.0 + rng.uniform(-0.1, 0.1))
    img = ImageEnhance.Contrast(img).enhance(1.0 + rng.uniform(-0.1, 0.1))
    if rng.random() < 0.2:
        img = img.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.1, 0.3)))
    return img

In [ ]:
def convert_to_conversation(sample, train: bool = False, rng: random.Random | None = None):
    img = augment_image(sample["image"], rng) if train else sample["image"]
    return {"messages": [
        {"role": "user", "content": [
            {"type": "text", "text": sample["instruction"]},
            {"type": "image", "image": img},
        ]},
        {"role": "assistant", "content": [{"type": "text", "text": sample["text"]}]},
    ]}

converted_dataset = [convert_to_conversation(s, train=True, rng=_aug_rng) for s in dataset]

In [ ]:
from unsloth import get_chat_template
processor = get_chat_template(processor, "gemma-4")

## Train

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

# max_length: the longest page target in this corpus (full table + title + letterhead +
# footer, all embedded together) runs up to ~4035 chars, since it includes wider 9-col
# wholesale_retail tables alongside 6-col retail_only ones. A v1-era run measured a real
# char-to-token ratio for this Khmer/Latin-mixed JSON content (a 2093-char target -> 1272
# generated chars at max_new_tokens=1024, i.e. ~1.24 chars/token), which puts the worst
# case around ~3250 tokens for the target text alone, before image tokens + the
# instruction are added on top. 6144 leaves real headroom above that estimate (couldn't
# verify the exact ratio locally -- this env's installed transformers has a
# Gemma-tokenizer compat bug -- so this is a generous margin over an estimate, not a
# measured exact minimum).
trainer = SFTTrainer(
    model = model,
    train_dataset = converted_dataset,
    processing_class = processor.tokenizer,
    data_collator = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        max_grad_norm = 0.3,
        warmup_ratio = 0.03,
        # -1 / a real int, never None: TrainingArguments._validate_args unconditionally does
        # `max_steps > 0 and num_train_epochs > 0`, so None crashes with a TypeError the
        # moment it's compared. -1 is the library's actual "unset" sentinel for max_steps.
        max_steps = 10 if SMOKE_TEST else -1,
        # See the EPOCHS knob + its reasoning in the data-loading cell above.
        num_train_epochs = 1 if SMOKE_TEST else EPOCHS,
        learning_rate = 2e-4,
        logging_steps = 1,
        save_strategy = "steps",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 6144,
    ),
)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"{trainer_stats.metrics['train_runtime']:.1f}s used for training.")
print(f"Peak reserved memory = {used_memory} GB / {max_memory} GB.")
print(f"Peak reserved memory for training (LoRA) = {used_memory_for_lora} GB.")

In [ ]:
# Push the trained adapter to the Hub RIGHT NOW, before any of the slower eval/diagnostic
# cells below -- a Colab free-tier disconnect (inactivity timeout or session max duration)
# during those cells would otherwise lose this entire training run, since local
# save_pretrained alone doesn't survive past this VM being torn down.
# Requires an HF_TOKEN Colab secret (key icon in the left sidebar) with write access.
from google.colab import userdata

# Versioned by dataset version AND epoch count, so sweep runs (see EPOCHS above) are each
# independently addressable rather than silently overwriting one another or Runs 1-3's
# Soxavin/gemma4-e2b-ardb-lora (trained on v2 data).
_ADAPTER_REPO_ID = "Soxavin/gemma4-e2b-ardb-lora-v5-smoke" if SMOKE_TEST else f"Soxavin/gemma4-e2b-ardb-lora-v5-e{EPOCHS}"
_hf_token = userdata.get("HF_TOKEN")
model.push_to_hub(_ADAPTER_REPO_ID, token=_hf_token)
processor.push_to_hub(_ADAPTER_REPO_ID, token=_hf_token)
print(f"Pushed to https://huggingface.co/{_ADAPTER_REPO_ID}")

## Inference sanity check

In [ ]:
from transformers import TextStreamer

sample = val_dataset[0]
messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": sample["instruction"]}]}]
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(sample["image"], input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

text_streamer = TextStreamer(processor, skip_prompt=True)
_ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=5000,
                   use_cache=True, temperature=1.0, top_p=0.95, top_k=64)
print("\n--- expected ---\n", sample["text"])

## Evaluate on the validation split (per-label CER + bbox accuracy, not eyeballing)

Each prediction is a JSON list of regions, not one string, so CER against the whole blob
isn't meaningful. Instead: parse both the prediction and the reference, match regions by
`label` (both lists are sorted top-to-bottom, so positional zip within a label stays
correct), then report mean CER per label (skipping `Picture`, whose text is always empty)
plus a bbox accuracy signal (mean per-coordinate absolute difference, 0-1000 scale) across
all matched regions. Malformed JSON is tracked as its own failure count rather than
crashing the loop or silently being dropped.

In [ ]:
import json

def levenshtein(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb))
        prev = cur
    return prev[-1]

def cer(pred: str, ref: str) -> float:
    return levenshtein(pred, ref) / max(1, len(ref))

def parse_regions(text: str) -> list[dict] | None:
    try:
        regions = json.loads(text)
    except json.JSONDecodeError:
        return None
    return regions if isinstance(regions, list) else None

def generate(sample):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": sample["instruction"]}]}]
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(sample["image"], input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    # do_sample=False (greedy): eval numbers must be deterministic run-to-run to compare
    # smoke test vs. full run vs. later epochs. 5000: this corpus's longest page target
    # runs ~4035 chars, and Khmer tokenizes less efficiently than Latin text -- see
    # the max_length cell above for the char-to-token estimate this margin is based on.
    out = model.generate(**inputs, max_new_tokens=5000, use_cache=True, do_sample=False)
    return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

cer_by_label: dict[str, list[float]] = {}
bbox_diffs: list[float] = []
parse_failures = 0
count_mismatches: list[str] = []

# Each row here re-runs a full greedy generation (up to 5000 tokens, uncompiled model), so
# this loop can take a while with no output until it finishes -- print progress per row so
# it doesn't look stuck.
for i, s in enumerate(val_dataset):
    print(f"[{i + 1}/{len(val_dataset)}] generating doc_id={s['doc_id']} page={s['page']}...")
    expected = parse_regions(s["text"])
    predicted = parse_regions(generate(s))
    if predicted is None:
        parse_failures += 1
        continue

    exp_by_label: dict[str, list[dict]] = {}
    for r in expected:
        exp_by_label.setdefault(r["label"], []).append(r)
    pred_by_label: dict[str, list[dict]] = {}
    for r in predicted:
        if isinstance(r, dict) and "label" in r:
            pred_by_label.setdefault(r["label"], []).append(r)

    for label, exp_list in exp_by_label.items():
        pred_list = pred_by_label.get(label, [])
        if len(pred_list) != len(exp_list):
            count_mismatches.append(f"{s['doc_id']} p{s['page']} {label}: "
                                    f"expected {len(exp_list)}, got {len(pred_list)}")
        for exp_r, pred_r in zip(exp_list, pred_list):
            if label != "Picture":
                cer_by_label.setdefault(label, []).append(
                    cer(pred_r.get("text", ""), exp_r["text"]))
            exp_box, pred_box = exp_r.get("box_2d"), pred_r.get("box_2d")
            if isinstance(exp_box, list) and isinstance(pred_box, list) and len(exp_box) == len(pred_box) == 4:
                bbox_diffs.append(sum(abs(a - b) for a, b in zip(exp_box, pred_box)) / 4)

print(f"\nJSON parse failures: {parse_failures} / {len(val_dataset)}")
print(f"region count mismatches: {len(count_mismatches)}")
for msg in count_mismatches:
    print(f"  {msg}")
for label, cers in cer_by_label.items():
    print(f"{label}: mean CER over {len(cers)} matched rows = {sum(cers) / len(cers):.3f}")
if bbox_diffs:
    print(f"mean bbox coordinate abs diff (0-1000 scale): {sum(bbox_diffs) / len(bbox_diffs):.1f}")

In [ ]:
# Diagnostic: is Table's CER partly a truncation artifact (max_new_tokens cutting off a
# long combined JSON blob) rather than purely a content-accuracy issue? Compare the whole
# generated output's length against the expected target's length, then show each region's
# expected vs. generated text side by side so a length gap is visible per-region, not just
# in aggregate.
sample = next(s for s in val_dataset if any(r["label"] == "Table" for r in parse_regions(s["text"]) or []))
print(f"generating doc_id={sample['doc_id']} page={sample['page']}...")
pred_text = generate(sample)
print(f"expected length: {len(sample['text'])} chars")
print(f"generated length: {len(pred_text)} chars")

expected = parse_regions(sample["text"]) or []
predicted = parse_regions(pred_text) or []
pred_by_label: dict[str, list[dict]] = {}
for r in predicted:
    if isinstance(r, dict) and "label" in r:
        pred_by_label.setdefault(r["label"], []).append(r)

for r in expected:
    label = r["label"]
    matches = pred_by_label.get(label, [])
    got = matches[0]["text"] if matches else "<region missing from prediction>"
    print(f"\n--- {label} ---")
    print("expected: ", r["text"][:300])
    print("generated:", got[:300])

In [ ]:
# Diagnostic: 3/9 validation rows failed to parse as JSON last run. Print the RAW
# (unparsed) generated text for EVERY such row (not just the first) so we can tell whether
# they share the same pattern (e.g. a dropped region + unclosed list) or differ.
failures = []
for i, _s in enumerate(val_dataset):
    print(f"[{i + 1}/{len(val_dataset)}] generating doc_id={_s['doc_id']} page={_s['page']}...")
    _pred_text = generate(_s)
    if parse_regions(_pred_text) is None:
        failures.append((_s, _pred_text))

print(f"\n{len(failures)} parse failure(s) out of {len(val_dataset)} validation rows")
for _s, _pred_text in failures:
    print(f"\n=== doc_id={_s['doc_id']} page={_s['page']} ===")
    print(f"expected length: {len(_s['text'])} chars")
    print(f"generated length: {len(_pred_text)} chars")
    print("--- raw generated text (unparsed) ---")
    print(_pred_text)

## Save

In [ ]:
model.save_pretrained("gemma4_e2b_ardb_lora")
processor.save_pretrained("gemma4_e2b_ardb_lora")
# model.push_to_hub("your_name/gemma4_e2b_ardb_lora", token="YOUR_HF_TOKEN")
# processor.push_to_hub("your_name/gemma4_e2b_ardb_lora", token="YOUR_HF_TOKEN")